# 有机锂中间体稳定性预测：从原始数据到反应器推荐

**目标**: 输入 SMILES + 温度 → 推荐 flash / flow / batch 反应器

**流程**:
1. 读取原始数据 (yield vs tR at multiple T)
2. 动力学拟合 → k_f, k_d, t_max
3. Arrhenius 拟合 → Ea, lnA
4. 计算化学描述符 (xTB)
5. 描述符筛选 + 模型验证
6. 反应器分类工具

**数据来源**: Nagaki/Yoshida 等 12 篇流动化学文献 [De Gennaro 2014]

## Step 1: 读取原始数据

原始数据在 `dataset-manual-corrected.numbers` 中，包含 2610 行 yield-vs-tR 数据。
每一行是一个实验点：在温度 T、停留时间 tR 下测得的产率 yield%。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import LeaveOneOut
from rdkit import Chem
from rdkit.Chem import Draw, Descriptors
from collections import defaultdict
from itertools import combinations
import json, warnings, os
warnings.filterwarnings('ignore')

# ===== 1.1 从 .numbers 文件读取原始数据 =====
# 如果没有 numbers_parser，可以用已导出的 CSV
DATA_DIR = os.path.dirname(os.path.abspath('__file__')) if '__file__' in dir() else '.'
NUMBERS_FILE = os.path.join(DATA_DIR, 'dataset-manual-corrected.numbers')
CSV_FILE = os.path.join(DATA_DIR, 'clean_organolithium_unified_descriptors.csv')

try:
    import numbers_parser
    doc = numbers_parser.Document(NUMBERS_FILE)
    table = doc.sheets[0].tables[0]
    headers = [str(table.cell(0, c).value) for c in range(table.num_cols)]
    rows = []
    for r in range(1, table.num_rows):
        row = {}
        for c in range(table.num_cols):
            val = table.cell(r, c).value
            row[headers[c]] = val
        rows.append(row)
    df = pd.DataFrame(rows)
    print(f"从 .numbers 读取: {df.shape}")
except Exception as e:
    print(f".numbers 读取失败 ({e}), 使用 CSV 备份")
    df = pd.read_csv(CSV_FILE)
    print(f"从 CSV 读取: {df.shape}")

# 类型转换
for col in ['tR1_s', 'T1_C', 'tR2_s', 'T2_C', 'yield_pct']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print(f"\n列名: {list(df.columns)}")
print(f"\n化合物数: {df['intermediate_smiles_canonical'].nunique()}")
print(f"论文数: {df['paper_id'].nunique()}")
print(f"温度范围: {df['T1_C'].min():.0f} ~ {df['T1_C'].max():.0f} °C")
print(f"tR 范围: {df['tR1_s'].min():.4f} ~ {df['tR1_s'].max():.1f} s")
print(f"\n类别分布:")
print(df['intermediate_class'].value_counts())

## Step 2: 数据可视化 — yield vs tR 曲线

有机锂中间体的动力学可以用**竞争生成+分解模型**描述：

$$\text{yield}(t_R) = y_{\max} \times (1 - e^{-k_f \cdot t_R}) \times e^{-k_d \cdot t_R}$$

- $k_f$: 生成速率 (s⁻¹) — 卤素-金属交换
- $k_d$: 分解速率 (s⁻¹) — 中间体热分解
- $t_{\max} = \frac{\ln(k_f/k_d)}{k_f - k_d}$: 最优停留时间 = 产率峰值位置

In [ ]:
# ===== 2.1 选几个典型化合物画 yield vs tR =====
# 只用甲醇淬灭的数据 (最可靠)
methanol = df[df['electrophile'].str.contains('methanol', na=False, case=False)]

# 选 3 个代表: 一个稳定、一个中等、一个不稳定
examples = {
    'p-NO₂-PhLi (不稳定)': '[Li]c1ccc([N+](=O)[O-])cc1',
    'tBu o-LiB (中等)': '[Li]c1ccccc1C(=O)OC(C)(C)C',
    'p-Br-PhLi (稳定)': '[Li]c1ccc(Br)cc1',
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors = plt.cm.coolwarm(np.linspace(0, 1, 10))

for ax, (label, smi) in zip(axes, examples.items()):
    sub = methanol[methanol['intermediate_smiles_canonical'] == smi]
    if len(sub) == 0:
        sub = df[df['intermediate_smiles_canonical'] == smi]
    
    temps = sorted(sub['T1_C'].dropna().unique())
    clrs = plt.cm.coolwarm(np.linspace(0, 1, len(temps)))
    
    for T, c in zip(temps, clrs):
        t_data = sub[sub['T1_C'] == T].sort_values('tR1_s')
        ax.scatter(t_data['tR1_s'], t_data['yield_pct'], c=[c], s=20, label=f'{T:.0f}°C')
    
    ax.set_xscale('log')
    ax.set_xlabel('tR₁ (s)')
    ax.set_ylabel('Yield (%)')
    ax.set_title(label, fontsize=10)
    ax.legend(fontsize=7, loc='best')
    ax.set_ylim(-5, 105)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.suptitle('原始数据: yield vs tR（按温度着色）', fontsize=12, y=1.02)
plt.show()

print("观察:")
print("  左: 高温(红色)产率快速上升又快速下降 → 不稳定")
print("  中: 低温产率平坦, 高温才出现衰减 → 中等稳定")
print("  右: 所有温度产率平坦 → 在实验窗口内很稳定")

## Step 3: 动力学拟合 — 提取 k_f, k_d, t_max

对每个 (化合物, 温度) 组合拟合竞争动力学模型，提取：
- **k_f**: 生成速率
- **k_d**: 分解速率 (如果可见衰减 ≥10%)
- **t_max**: 最优停留时间 = ln(k_f/k_d) / (k_f - k_d)
- **t½**: 半衰期 = ln(2) / k_d

**注意**: 甲醇淬灭数据优先（最可靠），同一 (compound, T) 下有多种 electrophile 时只取甲醇数据。

In [ ]:
# ===== 3.1 定义动力学模型 =====

def competing_kinetics(tR, k_f, k_d, y_max):
    """竞争生成+分解模型: yield = y_max * (1-exp(-k_f*tR)) * exp(-k_d*tR)"""
    with np.errstate(over='ignore'):
        return y_max * (1.0 - np.exp(-k_f * tR)) * np.exp(-k_d * tR)

def formation_only(tR, k_f, y_max):
    """仅生成模型 (无可见分解): yield = y_max * (1-exp(-k_f*tR))"""
    return y_max * (1.0 - np.exp(-k_f * tR))

def decay_only(tR, k_d, y0):
    """纯衰减模型 (从平台开始): yield = y0 * exp(-k_d*tR)"""
    return y0 * np.exp(-k_d * tR)

MIN_POINTS = 4      # 每组最少数据点
MIN_DECAY_DROP = 10  # 最小衰减量 (%)

def fit_curve(tR_arr, y_arr):
    """拟合一组 (tR, yield) 数据，返回动力学参数"""
    tR = np.array(tR_arr, dtype=float)
    y = np.array(y_arr, dtype=float)
    order = np.argsort(tR); tR = tR[order]; y = y[order]
    
    peak_idx = np.argmax(y)
    y_peak = y[peak_idx]
    tR_peak = tR[peak_idx]
    
    # 判断是否有可见衰减
    has_decay = (peak_idx < len(y)-1) and (y_peak - y[-1] >= MIN_DECAY_DROP)
    
    if has_decay:
        # 拟合竞争动力学 (3 参数)
        try:
            popt, _ = curve_fit(competing_kinetics, tR, y,
                p0=[1.0/max(tR_peak,1e-6), 0.1/max(tR_peak,1e-6), y_peak*1.1],
                bounds=([1e-6,1e-10,1.0], [1e8,1e6,150.0]), maxfev=10000)
            k_f, k_d, y_max = popt
            y_pred = competing_kinetics(tR, *popt)
            ss_res = np.sum((y-y_pred)**2); ss_tot = np.sum((y-np.mean(y))**2)
            r2 = 1-ss_res/ss_tot if ss_tot>0 else 0
            
            t_half = np.log(2)/k_d
            t_peak = np.log(k_f/k_d)/(k_f-k_d) if k_f>k_d else tR_peak
            
            if t_half > 1e6:  # k_d 太小，重新分类为 formation_only
                pass
            else:
                return {'model':'competing', 'k_f':k_f, 'k_d':k_d,
                        't_half':t_half, 't_peak':t_peak, 'y_max':y_max, 'r2':r2}
        except: pass
        
        # 备选: 只拟合衰减部分 (平台+慢衰减情况)
        if peak_idx > 0:
            tR_d = tR[peak_idx:]; y_d = y[peak_idx:]
            if len(tR_d) >= 3:
                tR_s = tR_d - tR_d[0]
                try:
                    popt_d, _ = curve_fit(decay_only, tR_s, y_d,
                        p0=[0.1, y_peak], bounds=([1e-10,1.0],[1e6,150.0]), maxfev=10000)
                    k_d_p, y0 = popt_d
                    t_half_p = np.log(2)/k_d_p
                    if t_half_p < 1e6:
                        return {'model':'competing', 'k_f':1e6, 'k_d':k_d_p,
                                't_half':t_half_p, 't_peak':tR_peak, 'y_max':y0, 'r2':0.5}
                except: pass
    
    # 无可见衰减 → formation_only
    try:
        popt, _ = curve_fit(formation_only, tR, y,
            p0=[1.0/max(tR_peak,1e-6), y_peak*1.1],
            bounds=([1e-6,1.0],[1e8,150.0]), maxfev=10000)
        k_f, y_max = popt
        y_pred = formation_only(tR, *popt)
        ss_res = np.sum((y-y_pred)**2); ss_tot = np.sum((y-np.mean(y))**2)
        r2 = 1-ss_res/ss_tot if ss_tot>0 else 0
        return {'model':'formation_only', 'k_f':k_f, 'k_d':None,
                't_half':None, 't_peak':None, 'y_max':y_max, 'r2':r2}
    except:
        return None

print("动力学模型定义完成 ✓")
print("  competing_kinetics: yield = y_max × (1-exp(-k_f·tR)) × exp(-k_d·tR)")
print("  formation_only:     yield = y_max × (1-exp(-k_f·tR))")
print("  decay_only:         yield = y0 × exp(-k_d·tR)")

In [ ]:
# ===== 3.2 对所有 (compound, T) 组合拟合 =====

# 只用 tR1 数据
tr1_data = df[df['tR_step'].str.startswith('tR1', na=False)].copy()

# 按 (SMILES, T, electrophile) 分组
groups = defaultdict(list)
elec_sub = defaultdict(lambda: defaultdict(list))

for _, r in tr1_data.iterrows():
    smi = r.get('intermediate_smiles_canonical', '')
    T_C = r.get('T1_C', np.nan)
    tR = r.get('tR1_s', np.nan)
    y = r.get('yield_pct', np.nan)
    if not smi or np.isnan(T_C) or np.isnan(tR) or np.isnan(y):
        continue
    elec = str(r.get('electrophile', 'unknown')).strip() or 'unknown'
    source = str(r.get('data_source_type', '')).strip()
    
    point = {'tR': tR, 'yield': y, 'intermediate': r.get('intermediate',''), 'paper': str(r.get('paper_id',''))[:40]}
    groups[(smi, T_C)].append(point)
    elec_sub[(smi, T_C)][(elec, source)].append(point)

# 甲醇优先：如果有多种 electrophile，用甲醇数据
for (smi, T_C), sub_dict in elec_sub.items():
    if len(sub_dict) <= 1:
        continue
    methanol_pts = []
    for (elec, src), pts in sub_dict.items():
        if 'methanol' in elec.lower():
            methanol_pts.extend(pts)
    if len(methanol_pts) >= MIN_POINTS:
        groups[(smi, T_C)] = methanol_pts

print(f"总 (compound, T) 组合: {len(groups)}")

# 拟合每组
results = []
n_decay, n_form, n_skip = 0, 0, 0

for (smi, T_C), points in sorted(groups.items()):
    if len(points) < MIN_POINTS:
        n_skip += 1; continue
    
    tR_arr = [p['tR'] for p in points]
    y_arr = [p['yield'] for p in points]
    name = points[0]['intermediate']
    
    result = fit_curve(tR_arr, y_arr)
    
    # 如果合并拟合失败但有多种 electrophile，尝试单 electrophile
    if (result is None or result['model'] == 'formation_only') and len(elec_sub[(smi,T_C)]) > 1:
        for (elec, src), sub_pts in elec_sub[(smi,T_C)].items():
            if len(sub_pts) < MIN_POINTS: continue
            sub_result = fit_curve([p['tR'] for p in sub_pts], [p['yield'] for p in sub_pts])
            if sub_result and sub_result['model'] == 'competing':
                result = sub_result; break
    
    if result is None:
        n_skip += 1; continue
    
    results.append({
        'intermediate': name, 'intermediate_smiles': smi,
        'T_C': T_C, 'T_K': T_C + 273.15,
        'model': result['model'], 'k_f': result['k_f'],
        'k_d': result.get('k_d'), 't_half_s': result.get('t_half'),
        't_peak_s': result.get('t_peak'), 'y_max': result.get('y_max'),
        'fit_r2': result.get('r2'),
    })
    if result['model'] == 'competing': n_decay += 1
    else: n_form += 1

halflives = pd.DataFrame(results)
print(f"\n拟合完成:")
print(f"  有衰减 (competing): {n_decay}")
print(f"  纯生成 (formation_only): {n_form}")
print(f"  跳过: {n_skip}")
print(f"  化合物数: {halflives['intermediate_smiles'].nunique()}")
halflives.head()

## Step 4: Arrhenius 拟合 — 提取 Ea, lnA

对有 ≥2 个温度 k_d 数据的化合物做 Arrhenius 拟合：

$$\ln(k_d) = \ln(A) - \frac{E_a}{RT}$$

斜率 = -Ea/R → Ea (活化能)，截距 = lnA (前指因子)

In [ ]:
# ===== 4.1 Arrhenius 拟合 =====
R_gas = 8.314e-3  # kJ/(mol·K)

decay_data = halflives[halflives['model'] == 'competing'].copy()
decay_data['k_d'] = decay_data['k_d'].astype(float)

arrhenius_results = []
for smi, g in decay_data.groupby('intermediate_smiles'):
    name = g['intermediate'].iloc[0]
    g = g.sort_values('T_K')
    
    # 单调性过滤: k_d 应随温度单调递增
    idx = [0]
    for i in range(1, len(g)):
        if g.iloc[i]['k_d'] > g.iloc[idx[-1]]['k_d']:
            idx.append(i)
    gc = g.iloc[idx]
    n_T = len(gc)
    if n_T < 2: continue
    
    T_K = gc['T_K'].values; k_d = gc['k_d'].values
    coeffs = np.polyfit(1.0/T_K, np.log(k_d), 1)
    Ea = -coeffs[0] * R_gas; lnA = coeffs[1]
    
    pred = np.polyval(coeffs, 1.0/T_K)
    ss_r = np.sum((np.log(k_d)-pred)**2); ss_t = np.sum((np.log(k_d)-np.mean(np.log(k_d)))**2)
    r2 = 1-ss_r/ss_t if ss_t > 0 else 1.0
    
    # 质量分级
    if n_T >= 3 and r2 >= 0.85 and Ea > 0: tier = 'tier1'
    elif n_T >= 3: tier = 'tier2_low_r2'
    else: tier = 'tier2_2pt'
    
    arrhenius_results.append({
        'intermediate': name, 'intermediate_smiles': smi,
        'Ea_kJ_mol': round(Ea, 2), 'ln_A': round(lnA, 4),
        'arrhenius_r2': round(r2, 4), 'n_temperatures': n_T,
        'quality_tier': tier,
    })

arr_df = pd.DataFrame(arrhenius_results).sort_values('Ea_kJ_mol')

print(f"Arrhenius 拟合: {len(arr_df)} 个化合物")
print(f"  Tier1 (≥3T, R²≥0.85): {len(arr_df[arr_df.quality_tier=='tier1'])}")
print(f"  Tier2_2pt: {len(arr_df[arr_df.quality_tier=='tier2_2pt'])}")
print(f"  Tier2_low_r2: {len(arr_df[arr_df.quality_tier=='tier2_low_r2'])}")
print(f"\nEa 范围: {arr_df.Ea_kJ_mol.min():.1f} ~ {arr_df.Ea_kJ_mol.max():.1f} kJ/mol")

# Ea-lnA 补偿效应
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(arr_df['Ea_kJ_mol'], arr_df['ln_A'], c='#4a90d9', s=50, edgecolors='black', linewidth=0.5)
z = np.polyfit(arr_df['Ea_kJ_mol'], arr_df['ln_A'], 1)
x_fit = np.linspace(arr_df['Ea_kJ_mol'].min()-5, arr_df['Ea_kJ_mol'].max()+5)
ax.plot(x_fit, np.polyval(z, x_fit), 'r--', lw=2)
r_corr = np.corrcoef(arr_df['Ea_kJ_mol'], arr_df['ln_A'])[0,1]
ax.text(0.05, 0.95, f'r = {r_corr:.2f}', transform=ax.transAxes, fontsize=14, fontweight='bold', va='top')
ax.set_xlabel('Ea (kJ/mol)'); ax.set_ylabel('ln(A)')
ax.set_title('Finding 2: Ea-lnA 补偿效应\n高 Ea 不一定意味着更稳定')
plt.tight_layout(); plt.show()

arr_df[['intermediate', 'Ea_kJ_mol', 'ln_A', 'arrhenius_r2', 'n_temperatures', 'quality_tier']].head(10)

## Step 5: 描述符 — 已在数据集中 (xTB 级别)

原始数据集已包含 xTB 描述符（之前用 `fill_dft_descriptors.py` 计算）。这里直接使用：

| 描述符 | 列名 | 物理意义 |
|---|---|---|
| q(C_ipso) | dft_charge_C_ipso | C-Li 碳上的 Mulliken 电荷 → 碳负离子稳定性 |
| d(Li-C) | dft_LiC_bond_A | Li-C 键长 (Å) → 键强度 |
| BDE | dft_LiC_BDE_kJ | 键离解能 (kJ/mol) → 热力学稳定性 |
| %Vbur | buried_vol_Li | Li 周围埋藏体积 → 空间位阻 |
| Gsolv | dft_Gsolv_kJ | THF 溶剂化自由能 → 溶剂效应 |
| HOMO | dft_HOMO_eV | 最高占据轨道能量 |
| η | HOMO_LUMO_gap_eV | 化学硬度 (LUMO-HOMO)/2 |
| fukui | fukui_f_minus_C | C_ipso 的 Fukui 函数 → 局部亲核性 |
| B1/B5/L | sterimol_B1/B5/L | Sterimol 取代基形状参数 |
| vol | mol_volume | 分子体积 (RDKit) |

如果需要重新计算描述符（比如新化合物），运行 `fill_dft_descriptors.py` 和 `fill_new_descriptors.py`。

In [ ]:
# ===== 5.1 提取每个化合物的描述符 (从原始数据去重) =====
desc_cols = {
    'q_C': 'dft_charge_C_ipso', 'd_LiC': 'dft_LiC_bond_A',
    'BDE': 'dft_LiC_BDE_kJ', '%Vbur': 'buried_vol_Li',
    'Gsolv': 'dft_Gsolv_kJ', 'HOMO': 'dft_HOMO_eV',
    'eta': 'HOMO_LUMO_gap_eV', 'fukui': 'fukui_f_minus_C',
    'B1': 'sterimol_B1', 'B5': 'sterimol_B5', 'L': 'sterimol_L',
    'vol': 'mol_volume', 'dipole': 'dft_dipole_D', 'q_Li': 'dft_charge_Li',
}

# 每个化合物取一行描述符 (去重)
unique_compounds = df.drop_duplicates(subset='intermediate_smiles_canonical')
desc_table = unique_compounds[['intermediate', 'intermediate_smiles_canonical', 'intermediate_class'] + 
                               list(desc_cols.values())].copy()

# 转为数值
for col in desc_cols.values():
    desc_table[col] = pd.to_numeric(desc_table[col], errors='coerce')

print(f"化合物描述符表: {len(desc_table)} 个化合物")
print(f"\n描述符覆盖率:")
for name, col in desc_cols.items():
    n = desc_table[col].notna().sum()
    print(f"  {name:10s} ({col:25s}): {n}/{len(desc_table)} ({n/len(desc_table)*100:.0f}%)")

# 也加载 DFT 层级的缓存数据 (如果有)
dft_caches = {}
for cache_name, cache_file in [('HF', 'hf_def2svp_cache.json'), ('M06-2X', 'm06_def2svp_cache.json'),
                                 ('ADCH', 'adch_cache.json'), ('QTAIM', 'qtaim_bcp_cache.json')]:
    path = os.path.join(DATA_DIR, cache_file)
    if os.path.exists(path):
        with open(path) as f:
            dft_caches[cache_name] = json.load(f)
        print(f"\n加载 {cache_name} 缓存: {len(dft_caches[cache_name])} 个化合物")
    else:
        print(f"\n{cache_name} 缓存不存在 ({cache_file})")

## Step 6: 描述符筛选 — 穷举 LOO-CV

对 Ea 和 t_max 两个目标，穷举所有 2-param 和 3-param 描述符组合，用 LOO-CV 评估。

**关键验证**:
- LOO-CV: 留一交叉验证
- 置换检验: 打乱 y 值 500 次，确认 R² 非偶然
- LOCO: 留一类交叉验证 → 检验跨类别泛化

In [ ]:
# ===== 6.1 合并描述符 + Ea → 模型筛选 =====

# 合并 Arrhenius 参数 和 描述符
tier1 = arr_df[arr_df['quality_tier'] == 'tier1']
model_data = desc_table.merge(tier1[['intermediate_smiles', 'Ea_kJ_mol', 'ln_A']],
                               left_on='intermediate_smiles_canonical',
                               right_on='intermediate_smiles', how='inner')

print(f"Tier-1 建模数据: {len(model_data)} 个化合物")

# LOO-CV R² 函数
def loo_r2(X, y):
    yp = np.zeros_like(y, dtype=float)
    for tr, te in LeaveOneOut().split(X):
        yp[te] = LinearRegression().fit(X[tr], y[tr]).predict(X[te])
    ss_r = np.sum((y-yp)**2); ss_t = np.sum((y-np.mean(y))**2)
    return 1 - ss_r/ss_t if ss_t > 0 else 0

# 穷举 2-param 和 3-param
desc_names = list(desc_cols.keys())
target = model_data['Ea_kJ_mol'].values

print(f"\n{'#p':>3s} {'描述符':35s} {'LOO-R²':>8s} {'n':>4s}")
print("="*55)

all_results = []
for np_ in [2, 3]:
    for combo in combinations(desc_names, np_):
        cols = [desc_cols[d] for d in combo]
        v = model_data[cols + ['Ea_kJ_mol']].dropna()
        if len(v) < max(5, len(model_data)*0.6): continue
        r2 = loo_r2(v[cols].values, v['Ea_kJ_mol'].values)
        all_results.append({'np': np_, 'desc': '+'.join(combo), 'r2': r2, 'n': len(v), 'cols': cols})

# Top 5 per #params
for np_ in [2, 3]:
    sub = sorted([r for r in all_results if r['np']==np_], key=lambda x: -x['r2'])
    for r in sub[:5]:
        star = ' ★' if r['r2'] > 0.5 else ''
        print(f"{r['np']:3d} {r['desc']:35s} {r['r2']:8.3f} {r['n']:4d}{star}")
    print()

# 最佳全局 Ea 模型
best_ea = max(all_results, key=lambda x: x['r2'])
print(f"最佳 Ea 模型: {best_ea['desc']} → LOO-R² = {best_ea['r2']:.3f}")

## Step 7: t_max 反应器分类模型

**核心创新**: 绕过 Ea-lnA 补偿问题，直接预测 t_max（最优停留时间）。

$$\log_{10}(t_{\max}) = w_1 \cdot q_C + w_2 \cdot d_{LiC} + w_3 \cdot L + w_4 \cdot \frac{1}{T} + b$$

分类边界（基于反应器设备能力）：
- **flash**: t_max < 0.1 s (微混合器)
- **flow**: 0.1 s < t_max < 60 s (管式反应器)
- **batch**: t_max > 60 s (常规操作)

In [ ]:
# ===== 7.1 构建 t_max 数据集 =====

# 从 halflives 中提取有 t_peak 的数据
exact_tmax = halflives[halflives['model'] == 'competing'].copy()
exact_tmax = exact_tmax[exact_tmax['t_peak_s'].notna() & (exact_tmax['t_peak_s'] > 0)]
exact_tmax['log_tmax'] = np.log10(exact_tmax['t_peak_s'].astype(float).clip(lower=1e-10))
exact_tmax['inv_T'] = 1.0 / (exact_tmax['T_C'].astype(float) + 273.15)

# 合并描述符
tmax_data = exact_tmax.merge(desc_table[['intermediate_smiles_canonical'] + list(desc_cols.values())],
                              left_on='intermediate_smiles', right_on='intermediate_smiles_canonical', how='inner')

# 转数值
for col in desc_cols.values():
    tmax_data[col] = pd.to_numeric(tmax_data[col], errors='coerce')

print(f"t_max 数据: {len(tmax_data)} 行 ({tmax_data['intermediate_smiles'].nunique()} 化合物)")

# ===== 7.2 穷举描述符 + 1/T → log(t_max) → 分类 =====
def classify_reactor(tmax):
    if tmax < 0.1: return 'flash'
    elif tmax < 60: return 'flow'
    else: return 'batch'

tmax_data['reactor'] = tmax_data['t_peak_s'].astype(float).apply(classify_reactor)
print(f"\n反应器分布: {tmax_data['reactor'].value_counts().to_dict()}")

# 搜索最佳 2desc + 1/T 和 3desc + 1/T
print(f"\n{'#d':>3s} {'描述符+1/T':40s} {'R²':>6s} {'分类准确率':>10s}")
print("="*65)

tmax_results = []
for np_ in [2, 3]:
    for combo in combinations(desc_names, np_):
        feats_cols = [desc_cols[d] for d in combo] + ['inv_T']
        v = tmax_data[feats_cols + ['log_tmax', 'reactor']].dropna()
        if len(v) < 30: continue
        
        X = v[feats_cols].values; y = v['log_tmax'].values
        # LOO predict
        yp = np.zeros(len(v))
        for tr, te in LeaveOneOut().split(X):
            yp[te] = LinearRegression().fit(X[tr], y[tr]).predict(X[te])
        
        ss_r = np.sum((y-yp)**2); ss_t = np.sum((y-np.mean(y))**2)
        r2 = 1-ss_r/ss_t if ss_t>0 else 0
        
        pred_r = [classify_reactor(10**p) for p in yp]
        actual_r = v['reactor'].values.tolist()
        acc = sum(p==a for p,a in zip(pred_r, actual_r)) / len(v)
        
        tmax_results.append({'np': np_, 'desc': '+'.join(combo)+'+1/T', 'r2': r2, 'acc': acc, 'n': len(v)})

for np_ in [2, 3]:
    sub = sorted([r for r in tmax_results if r['np']==np_], key=lambda x: -x['acc'])
    for r in sub[:3]:
        star = ' ★' if r['acc'] > 0.85 else ''
        print(f"{r['np']:3d} {r['desc']:40s} {r['r2']:6.3f} {r['acc']:10.1%}{star}")
    print()

best_tmax = max(tmax_results, key=lambda x: x['acc'])
print(f"最佳 t_max 分类模型: {best_tmax['desc']} → 准确率 = {best_tmax['acc']:.1%}")

## Step 8: 训练最终模型 + 可视化

In [ ]:
# ===== 8.1 用最佳模型训练并画图 =====

# 提取最佳描述符名 (去掉 +1/T)
best_desc_names = best_tmax['desc'].replace('+1/T','').split('+')
best_feat_cols = [desc_cols[d] for d in best_desc_names] + ['inv_T']

v = tmax_data[best_feat_cols + ['log_tmax', 'reactor', 'T_C', 'intermediate']].dropna().reset_index(drop=True)
X = v[best_feat_cols].values; y = v['log_tmax'].values

# LOO 预测
yp = np.zeros(len(v))
for tr, te in LeaveOneOut().split(X):
    yp[te] = LinearRegression().fit(X[tr], y[tr]).predict(X[te])

pred_r = [classify_reactor(10**p) for p in yp]
actual_r = v['reactor'].values.tolist()
acc = sum(p==a for p,a in zip(pred_r, actual_r)) / len(v)

# 训练最终模型
reg = LinearRegression().fit(X, y)

# ===== 画图 =====
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# (a) Parity plot (log-log)
correct_mask = np.array([p==a for p,a in zip(pred_r, actual_r)])
ax1.scatter(v['t_peak_s'].astype(float).values[correct_mask], (10**yp)[correct_mask],
           c='#4a90d9', s=30, alpha=0.7, edgecolors='#2c3e50', linewidth=0.4, label='Correct')
ax1.scatter(v['t_peak_s'].astype(float).values[~correct_mask], (10**yp)[~correct_mask],
           c='#e74c3c', s=40, alpha=0.8, edgecolors='#2c3e50', linewidth=0.4, label='Misclassified')
ax1.set_xscale('log'); ax1.set_yscale('log')
ax1.axhline(y=0.1, color='orange', ls='--', lw=1.5, alpha=0.7)
ax1.axvline(x=0.1, color='orange', ls='--', lw=1.5, alpha=0.7)
lims = [3e-5, 20]
ax1.plot(lims, lims, 'r--', lw=1.5, alpha=0.5)
ax1.set_xlim(lims); ax1.set_ylim(lims)
ax1.set_xlabel('Actual t_max (s)'); ax1.set_ylabel('Predicted t_max (s)')
ax1.text(0.05, 0.95, f'Accuracy = {acc:.1%}\nn = {len(v)}', transform=ax1.transAxes,
         fontsize=12, fontweight='bold', va='top', bbox=dict(facecolor='lightyellow', alpha=0.9))
ax1.legend(fontsize=9)
ax1.set_title('(a) Reactor classification (LOO-CV)')

# (b) Feature importance
X_std = (X - X.mean(0)) / X.std(0)
reg_std = LinearRegression().fit(X_std, y)
imp = np.abs(reg_std.coef_)
imp_norm = imp / imp.sum()
feat_labels = best_desc_names + ['1/T']
order = np.argsort(imp_norm)[::-1]

ax2.barh(range(len(feat_labels)), imp_norm[order], 
         color=['#e74c3c' if feat_labels[i]=='1/T' else '#4a90d9' for i in order], height=0.5)
ax2.set_yticks(range(len(feat_labels)))
ax2.set_yticklabels([feat_labels[i] for i in order])
ax2.invert_yaxis()
for i, (idx, val) in enumerate(zip(order, imp_norm[order])):
    sign = '+' if reg.coef_[idx] > 0 else '−'
    ax2.text(val+0.01, i, f'{val:.2f} ({sign})', va='center', fontsize=10, fontweight='bold')
ax2.set_xlabel('Importance')
ax2.set_title('(b) Feature importance')

# 模型方程
coef_str = ' '.join([f'{reg.coef_[i]:+.2f}·{feat_labels[i]}' for i in range(len(feat_labels))])
plt.suptitle(f'log₁₀(t_max) = {coef_str} {reg.intercept_:+.2f}', fontsize=11, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"\n最终模型方程:")
print(f"  log₁₀(t_max/s) = {' '.join([f'{reg.coef_[i]:+.2f}×{feat_labels[i]}' for i in range(len(feat_labels))])} {reg.intercept_:+.2f}")

## Step 9: 预测工具 — 输入 SMILES + 温度 → 反应器推荐

**注意**: 描述符需要从 xTB 计算（需要 tblite + morfeus）。如果这些包不可用，可以手动输入描述符值。

In [ ]:
# ===== 9.1 预测函数 =====

def predict_reactor(smiles, T_celsius, model=reg, feat_names=best_desc_names):
    """
    输入 SMILES + 温度 → 预测 t_max → 推荐反应器
    
    返回: dict with t_max, reactor, descriptors
    """
    # 从数据集中查找描述符 (如果化合物已在数据集中)
    match = desc_table[desc_table['intermediate_smiles_canonical'] == smiles]
    
    if len(match) > 0:
        row = match.iloc[0]
        desc_values = [float(row[desc_cols[d]]) for d in feat_names]
    else:
        # TODO: 用 tblite + morfeus 从 SMILES 计算描述符
        # 这里先返回错误提示
        return {'error': f'SMILES {smiles} 不在数据集中，需要计算描述符'}
    
    inv_T = 1.0 / (T_celsius + 273.15)
    X_pred = np.array([desc_values + [inv_T]])
    
    log_tmax = model.predict(X_pred)[0]
    tmax = 10**log_tmax
    
    reactor = classify_reactor(tmax)
    
    return {
        'smiles': smiles,
        'T_celsius': T_celsius,
        't_max_s': round(tmax, 4),
        'reactor': reactor,
        'descriptors': dict(zip(feat_names, desc_values)),
    }

# ===== 9.2 演示预测 =====
test_cases = [
    ('[Li]c1ccc([N+](=O)[O-])cc1', -40, 'p-NO₂-PhLi'),
    ('[Li]c1ccccc1C(=O)OC(C)(C)C', -40, 'tBu o-LiB'),
    ('[Li]c1ccc([N+](=O)[O-])cc1', 0, 'p-NO₂-PhLi'),
    ('[Li]CCCC1CO1', -40, '3-(oxiran-2-yl)propylLi'),
    ('[Li]c1ccccc1I', -60, 'o-I-PhLi'),
    ('[Li]c1ccc(Br)cc1', 0, 'p-Br-PhLi'),
]

print(f"{'化合物':25s} {'T':>5s} {'t_max':>10s} {'反应器':>8s}")
print("="*55)
for smi, T, name in test_cases:
    result = predict_reactor(smi, T)
    if 'error' in result:
        print(f"  {name:25s} {T:>5d}°C {'?':>10s} {'?':>8s}  ({result['error'][:30]})")
    else:
        print(f"  {name:25s} {T:>5d}°C {result['t_max_s']:>10.4f}s {result['reactor']:>8s}")

## Step 10: 总结

### 6 个发现

| # | 发现 | 证据 | 图 |
|---|---|---|---|
| 1 | 机制异质性 | LOCO R² < -12 | finding1_LOCO.png |
| 2 | Ea-lnA 补偿 | r = 0.94 | finding2_EalnA_compensation.png |
| 3 | xTB > DFT | 28 描述符穷举 | ppt_fig1_full_correlation_matrix.png |
| 4 | TS 不可行 | 气相高估 3.8× | finding4_5_TS_entropy.png |
| 5 | ΔS‡ 区分机制 | 缔合(-121) vs 解离(+77) | finding4_5_TS_entropy.png |
| 6 | t_max 反应器工具 | 89% 准确率 | ppt_fig2_model_performance.png |

### 三层结构

```
科学层 (WHY)                解释层 (WHY NOT ab initio)    工程层 (HOW TO USE)
Finding 1: 机制异质性         Finding 4: TS 高估 3.8×       Finding 6: t_max 工具
Finding 2: Ea-lnA 补偿        Finding 5: ΔS‡ 区分机制       89% 准确率
Finding 3: xTB > DFT                                      flash/flow/batch
```

### 核心结论

*Activation parameters (Ea, lnA) provide mechanistic insight into the decomposition process, whereas t_max directly determines optimal operating conditions in flow systems. The combination of mechanistic analysis and data-driven prediction forms a complete framework from molecular understanding to process design.*

### 参考文献

- [1] De Gennaro 2014, *Lithium Compounds in Organic Synthesis*, Ch.18
- [7] Ramachandran 2010, *J. Phys. Chem. A*, 114, 8423
- [8] Bannwarth 2019, *J. Chem. Theory Comput.*, 15, 1652 (GFN2-xTB)
- [14] Collum 2007, *Angew. Chem. Int. Ed.*, 46, 3002